# N12 · Online Softmax 算法 walkthrough

**关联 lab**: L01.7

**学习目标**: 把 Triton fused softmax 的核心数学骨架（online softmax）从公式变成可断言的 numpy 复现。理解 `running_max` 平移、`running_sum` 重缩放、tail mask 的必要性，再去写真正的 .triton 实现时不会被数学搞糊涂。

**No-GPU 可完成度**: 100%。

**对应 MiniInfra**: `mini_infra/gpu/triton_softmax.py`

**对应真实源码**: `github_repo/triton/python/tutorials/02-fused-softmax.py`


## 1. Stable softmax 一次性版（ground truth）

数值稳定 softmax 的标准算法：

$$y_i = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

减去 `max(x)` 是为了避免 `exp` 溢出。这个版本一次性把整行加载进来，作为后续 online 版本的对照。

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import math
from mini_infra.gpu.triton_softmax import stable_softmax, online_softmax, max_abs_error, mask_needed

row = [-2.0, -1.0, 0.5, 0.5, 3.0, 100.0, -50.0, 0.0]
ref = stable_softmax(row)
print('row =', row)
print('softmax =', [round(v, 6) for v in ref])
print('sum     =', round(sum(ref), 6), '  (应当 == 1.0)')

## 2. Online softmax：块累加版

Triton kernel 实际跑的是 **online** softmax：每次只把一个 BLOCK_SIZE 的子块加载进 SMEM，逐块更新 (running_max, running_sum)。最后一次性输出。

关键不变量（如果省略会数值崩塌）：

$$\text{new\_max} = \max(\text{running\_max}, \text{block\_max})$$

$$\text{running\_sum} \leftarrow \text{running\_sum} \cdot e^{\text{running\_max} - \text{new\_max}} + \sum e^{\text{block} - \text{new\_max}}$$

中间这一步 `running_sum * exp(running_max - new_max)` 是 **necessary**——如果省略，旧的指数项会被低估，导致下一块累加错误。

In [ ]:
candidate = online_softmax(row, block_size=4)
print('online softmax (BLOCK=4) =', [round(v, 6) for v in candidate])
print('max_abs_err vs stable    =', max_abs_error(ref, candidate))
print()
for bs in [1, 2, 4, 8, 16]:
    out = online_softmax(row, block_size=bs)
    err = max_abs_error(ref, out)
    print(f'BLOCK={bs:>2}  max_abs_err = {err:.2e}')

**观察**: 不管 BLOCK_SIZE 是多少，max_abs_err 都应该 < 1e-12（fp64 / Python float）。这就是 online softmax 与 stable softmax 数学等价的证据。

## 3. 反证：删掉 running_sum 重缩放会怎样

为了让你亲眼看到崩塌，下面写一个**故意错误**的 online softmax，省掉 `running_sum * exp(running_max - new_max)` 这一步：

In [ ]:
def buggy_online_softmax(values, block_size):
    values = list(values)
    running_max = -math.inf
    running_sum = 0.0
    for start in range(0, len(values), block_size):
        block = values[start:start + block_size]
        block_max = max(block) if block else -math.inf
        new_max = max(running_max, block_max)
        # ❌ BUG: 漏了 `running_sum *= exp(running_max - new_max)`
        running_sum = running_sum + sum(math.exp(v - new_max) for v in block)
        running_max = new_max
    return [math.exp(v - running_max) / running_sum for v in values]

buggy = buggy_online_softmax(row, block_size=4)
print('buggy   =', [round(v, 6) for v in buggy])
print('stable  =', [round(v, 6) for v in ref])
print('sum buggy =', round(sum(buggy), 6), '  (应当 == 1.0，但因为漏重缩放而偏)')

看到结果偏了。在 fp64 上偏的可能小，但在 fp16 长序列上这个 bug 会让 max_abs_err 跳到 1e-2 量级，模型 loss 直接崩。

**记住**: Triton kernel 内对应代码：
```python
row_max = tl.maximum(row_max, block_max)
row_sum = row_sum * tl.exp(prev_max - row_max) + tl.sum(tl.exp(block - row_max))
```
中间那个 `tl.exp(prev_max - row_max)` 是不可省略的。

## 4. Tail mask 的必要性

当 `seq_len` 不是 BLOCK_SIZE 整数倍时，最后一个 block 必须用 mask 把超出 seq_len 的位置填 `-inf`，否则 `online_max` 会被垃圾值污染。

Triton kernel 中：
```python
col_offsets = tl.arange(0, BLOCK_SIZE)
mask = col_offsets < n_cols
x = tl.load(in_ptr + col_offsets, mask=mask, other=-float('inf'))
```

用 `mask_needed` 判定：

In [ ]:
for seq_len in [1024, 4096, 4097, 16384, 16385]:
    for bs in [256, 512, 1024]:
        m = mask_needed(seq_len, bs)
        print(f'seq={seq_len:>5} BLOCK={bs:>4}  mask_needed = {m}')

**注意**: `other=-float('inf')` 不能写成 `other=0`。如果填 0，则 `exp(0)=1` 会污染 sum 分母，导致输出概率不归一。

## 5. fp16 vs fp32 累加器

在 GPU 上，weights 与中间值常用 fp16，但 `running_max` 与 `running_sum` 必须 fp32 累加。下面用纯 Python 模拟 fp16 量化看会出什么：

In [ ]:
import struct

def to_fp16(x):
    # 模拟 fp16 round-trip
    packed = struct.pack('e', x) if abs(x) < 65504 else struct.pack('e', math.copysign(65504, x))
    return struct.unpack('e', packed)[0]

def fp16_online_softmax(values, block_size):
    """running_sum/max 都强制 fp16，模拟错误的累加"""
    values = list(values)
    running_max = -math.inf
    running_sum = 0.0
    for start in range(0, len(values), block_size):
        block = values[start:start + block_size]
        block_max = max(block) if block else -math.inf
        new_max = to_fp16(max(running_max, block_max))
        running_sum = to_fp16(
            to_fp16(running_sum * to_fp16(math.exp(running_max - new_max)))
            + to_fp16(sum(to_fp16(math.exp(v - new_max)) for v in block))
        )
        running_max = new_max
    return [math.exp(v - running_max) / running_sum for v in values]

# 长序列 + 大值，让 fp16 不够
long_row = [(i % 13) * 0.5 for i in range(4096)]
ref_long = stable_softmax(long_row)
fp16_long = fp16_online_softmax(long_row, block_size=128)
print(f'fp16 累加器 max_abs_err = {max_abs_error(ref_long, fp16_long):.2e}')
print('结论：长序列下 fp16 累加误差显著大于 fp32。Triton kernel 必须保 running_max/sum 为 fp32。')

## 6. 自检问题（运行后回答）

1. 写出 online softmax 的伪代码，指出哪一行省略后会数值崩塌。
2. `tl.load(... other=?)` 在 softmax 中应该填什么？为什么不能是 0？
3. `seq_len=4097, BLOCK_SIZE=1024` 时，`mask_needed` 返回什么？少了 mask 会导致输出哪几位错？
4. 如果 weights 用 fp16，`running_max` 与 `running_sum` 应该是什么 dtype？为什么？
5. `mini_infra.gpu.triton_softmax.online_softmax` 的纯 Python 版本与 Triton 实际 kernel 在什么意义上等价？什么意义上不等价？

## 7. 与 lab 项目的对接

下一步去写 v1 Triton kernel 时，按以下顺序保你不犯 N11/N12 学过的错误：

1. **数值正确**: 与 `stable_softmax` 比对 max_abs_err < 1e-5（fp32）或 < 1e-3（fp16）
2. **mask 完备**: 用 `mask_needed(seq_len, BLOCK_SIZE)` 自检；`other=-float('inf')`
3. **累加器精度**: `running_max` 与 `running_sum` 用 fp32，即使输入是 fp16
4. **不省重缩放**: `running_sum * tl.exp(prev_max - row_max)` 不能漏
5. **BLOCK_SIZE 选择**: 用 `triton.next_power_of_2(n_cols)`；4090 上从 1024 起步

Smoke 命令：
```bash
torchrun --nproc_per_node=1 labs/l04_gpu_kernel/scripts/bench_softmax.py --seq 4097 --block 1024
```

记得带 4097（非整数倍）跑一次专门验证 mask。